# MSM 地上風（10m）の指定地点時系列を取得してCSV化

京大生存圏研究所の日別NetCDF（MSM-S、1日約140MB）から、指定地点の最寄り格子の風を取り出してCSVにする。
**このノートブック単体で動作する**（他の `.py` は不要）。

## 必要なライブラリ

| ライブラリ | 用途 | 動作確認バージョン |
|---|---|---|
| numpy | 風速・風向の計算 | 2.0.1 |
| pandas | 時系列・CSV出力 | 2.3.1 |
| xarray | NetCDFの読み込み・最寄り格子の抽出 | 2025.7.1 |
| netCDF4 | xarray のNetCDF読み込みエンジン | 1.7.2 |
| requests | NetCDFのダウンロード | 2.32.4 |
| matplotlib | 確認プロット | 3.10.0 |

未インストールの場合:

```bash
pip install numpy pandas xarray netCDF4 requests matplotlib
```

- 日本語フォントは macOS の `Hiragino Sans` を想定（他のOSでは、最初のセルのフォント名を変更する）。
- Python 3.9 以上を想定。

## データについて

- データ源: 京都大学生存圏研究所 気象庁データアーカイブ（**研究・教育目的**での利用に限る）
- 時刻は1時間ごと24点/日。00,03,...,21UTC は初期値（FH00=解析値）、それ以外は直前の初期値からの1〜2時間予報 → `is_analysis` 列で区別
- 前日分は 01:00 GMT 頃に公開される
- 5日分で約700MBをダウンロードする。NetCDFは地点を抜き出した後に削除する（`KEEP = True` で残せる）

In [ ]:
import datetime as dt
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import xarray as xr

# 日本語フォント（macOS）。既定の DejaVu Sans では日本語が豆腐になる
matplotlib.rcParams['font.family'] = ['Hiragino Sans', 'Hiragino Maru Gothic Pro', 'AppleGothic', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止

## 取得・抽出の関数

In [ ]:
BASE_URL = "http://database.rish.kyoto-u.ac.jp/arch/jmadata/data/gpv/netcdf/MSM-S"
DATA_DIR = Path.cwd() / "data"   # ノートブックの作業フォルダ直下の data/
HEADERS = {"User-Agent": "tenkizu-msm-wind-fetch (research use)"}
REQUEST_INTERVAL_SEC = 3.0

# 既定の地点: 沖縄本島南部
DEFAULT_LAT = 26.1708
DEFAULT_LON = 127.7418


def day_url(day: dt.date) -> str:
    return f"{BASE_URL}/{day.year}/{day:%m%d}.nc"


def download_day(day: dt.date, dest: Path) -> bool:
    """1日分のNetCDFをダウンロードする。未公開（404）なら False。"""
    tmp = dest.with_suffix(".part")
    try:
        with requests.get(day_url(day), headers=HEADERS, stream=True, timeout=120) as r:
            if r.status_code == 404:
                return False
            r.raise_for_status()
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    except requests.RequestException:
        tmp.unlink(missing_ok=True)
        raise
    tmp.rename(dest)
    return True


def extract_point(nc_path: Path, lat: float, lon: float) -> pd.DataFrame:
    """NetCDF から最寄り格子の u, v を取り出し、風速・風向・解析値フラグを付ける。"""
    with xr.open_dataset(nc_path) as ds:
        pt = ds[["u", "v"]].sel(lat=lat, lon=lon, method="nearest").load()
        grid_lat, grid_lon = float(pt.lat), float(pt.lon)
    df = pt.to_dataframe()[["u", "v"]].reset_index()
    df["time_utc"] = df["time"]
    df["time_jst"] = df["time_utc"] + pd.Timedelta(hours=9)
    df["wspd"] = np.hypot(df["u"], df["v"])
    df["wdir"] = np.degrees(np.arctan2(-df["u"], -df["v"])) % 360
    df["is_analysis"] = df["time_utc"].dt.hour % 3 == 0   # FH00=初期値（解析値）
    df.attrs["grid"] = (grid_lat, grid_lon)
    return df[["time_utc", "time_jst", "u", "v", "wspd", "wdir", "is_analysis"]]


def fetch_range(start: dt.date, end: dt.date, lat: float, lon: float,
                keep: bool = False, data_dir: Path = DATA_DIR) -> tuple[pd.DataFrame, tuple[float, float] | None]:
    """start〜end（UTC日付, 両端含む）を取得して縦に連結する。"""
    data_dir.mkdir(parents=True, exist_ok=True)
    frames, grid = [], None
    days = pd.date_range(start, end, freq="D").date
    downloaded_prev = False
    for day in days:
        local = data_dir / f"{day:%m%d}.nc"
        existed = local.exists()
        if not existed:
            if downloaded_prev:
                time.sleep(REQUEST_INTERVAL_SEC)
            print(f"  {day}: ダウンロード中 {day_url(day)}", flush=True)
            if not download_day(day, local):
                print(f"  {day}: 未公開（404）のためスキップ")
                downloaded_prev = False
                continue
            downloaded_prev = True
        else:
            print(f"  {day}: ローカルの {local.name} を使用")
            downloaded_prev = False
        df = extract_point(local, lat, lon)
        grid = df.attrs.get("grid", grid)
        frames.append(df)
        if not existed and not keep:
            local.unlink()
    if not frames:
        return pd.DataFrame(), grid
    out = pd.concat(frames, ignore_index=True).drop_duplicates("time_utc")
    return out.sort_values("time_utc").reset_index(drop=True), grid


def default_end() -> dt.date:
    """UTC で昨日（前日分は 01:00 GMT 頃に公開される）。"""
    return dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)


In [ ]:
def default_end() -> dt.date:
    """UTC で昨日（前日分は 01:00 GMT 頃に公開される）。"""
    return dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)

## パラメータ

In [ ]:
# 地点（既定: 沖縄本島南部）
LAT, LON = 26.1708, 127.7418

# 取得期間（UTC日付, 'YYYY-MM-DD'）
START_DATE = None   # 例: '2026-09-28'。None なら END_DATE から DAYS 日前
END_DATE = None     # 例: '2026-10-02'。None なら昨日（UTC）
DAYS = 5            # START_DATE が None のときの日数

# True ならダウンロードしたNetCDFを data/ に残す
KEEP = False

END = dt.date.fromisoformat(END_DATE) if END_DATE else default_end()
START = dt.date.fromisoformat(START_DATE) if START_DATE else END - dt.timedelta(days=DAYS - 1)
assert START <= END, 'START_DATE が END_DATE より後になっている'
print(START, '〜', END, f'({(END - START).days + 1}日)', LAT, LON)

## 取得（1日ずつダウンロード→地点を抜き出し→NetCDF削除）
5日分で約700MBをダウンロードする。

In [ ]:
df, grid = fetch_range(START, END, LAT, LON, keep=KEEP)
print(f'最寄り格子: N{grid[0]:.4f}, E{grid[1]:.4f}  / {len(df)}行')
df.head()

## CSVに保存

In [ ]:
out = DATA_DIR / f'msm_wind_{START:%Y%m%d}_{END:%Y%m%d}.csv'
df.round(3).to_csv(out, index=False)
print(out)

## 確認プロット（● = 解析値）

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
an = df[df['is_analysis']]
ax1.plot(df['time_jst'], df['wspd'], lw=1, color='tab:blue', label='全時刻（予報含む）')
ax1.plot(an['time_jst'], an['wspd'], 'o', color='tab:red', label='解析値(FH00)')
ax1.set_ylabel('風速 (m/s)'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(df['time_jst'], df['wdir'], '.', color='tab:blue')
ax2.plot(an['time_jst'], an['wdir'], 'o', color='tab:red')
ax2.set_ylabel('風向 (度)'); ax2.set_ylim(0, 360); ax2.set_yticks(range(0, 361, 90)); ax2.grid(alpha=0.3)
ax2.set_xlabel('時刻 (JST)')
fig.suptitle(f'MSM 地上風  N{grid[0]:.2f}, E{grid[1]:.2f}')
fig.autofmt_xdate(); plt.show()